In [ ]:
# analysis 2
import os
import glob
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import neuroimage_analysis as na
from tqdm import tqdm

---

## Analysis 2: Does sLNM Accurately Recover Ground Truth?

This analysis asks whether sLNM can accurately recover the underlying disease network from lesion-symptom data.

**Simulated datasets**

100 lesions (4-mm radius spheres, randomly sampled) were held fixed across 1,200 datasets (300 Yeo–Schaefer ground-truth networks × 4 effect sizes). Symptoms ($S$) are a linear function of spatial similarity ($R$; Fisher-z Pearson's r) between each lesion FC map and the ground-truth network, plus noise ($\epsilon$), then z-scored:

$$S = \theta R + \epsilon, \quad \epsilon \sim \mathcal{N}(0, \sigma^2), \quad \eta^2 = \frac{\theta^2}{\theta^2 + \sigma^2}$$

Effect sizes: $\eta^2 = 0.0$ (random), $0.3$ (realistic), $0.6$ (strong), $0.99$ (deterministic).

**Recovery accuracy**

Assessed two ways: 

1. spatial similarity (Pearson's r) between each ground-truth network and its sLNM map. This measures global similarity between the two maps

2. alignment of both maps with the first three GSP1000 connectome PCs (absolute Fisher-z spatial correlation), the allows us to visualize how sLNM transform ground-truth map in the latent space of the connectome.

Monotonic trends across effect sizes were tested with the Jonckheere–Terpstra test (10,000 permutations). 



In [ ]:
dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(dir)
brain_template = nib.load(os.path.join(dir, "data/templates/Taylor_NHB_MNI152_T1_2mm_brain_mask_dil.nii.gz"))
brain_mask = brain_template.get_fdata() > 0

# Ground truth Schaefer 300 FC maps
schaefer_dir = os.path.join(dir, "data/schaefer_300_fcmap")
schaefer_files = sorted(glob.glob(os.path.join(schaefer_dir, '*.nii.gz')))
print(f'Found {len(schaefer_files)} Schaefer region FC maps for ground truth')

# Simulated lesion FC maps
lesion_dir = os.path.join(dir, "data/sim_lesions")
lesion_files = sorted(glob.glob(os.path.join(lesion_dir, 'lesion*AvgR.nii.gz')))
print(f'Found {len(lesion_files)} simulated lesion FC maps')

outdir = os.path.join(dir, "results")

pca_dir = os.path.join(dir, "data//pca")
pca_files = sorted(glob.glob(os.path.join(pca_dir, '*pca_voxelwise*nii.gz')))
pc1_3 = []
for i in range(1, 4):
    file = os.path.join(pca_dir, f'pca_voxelwise_pc{i}.nii.gz')
    pc1_3.append(file)



In [ ]:
effect_sizes = [0.0, 0.3, 0.6, 0.99]
n = 100
recover_results = {}

for effect_size in effect_sizes:
    gt_slnm = []
    gt_pc = {j: [] for j in range(1, 6)}
    slnm_pc = {j: [] for j in range(1, 6)}

    for i in tqdm(range(len(schaefer_files)), desc=f"effect_size={effect_size}"):
        ground_truth_nii = na.nifti_getdata(schaefer_files[i])
        dataset_i = na.gen_dataset(subject_maps=lesion_files,
                                ground_truth_maps=schaefer_files,
                                sample_size=n,
                                effect_size=effect_size,
                                ground_truth_seed=i,
                                z_transform = True,
                                lesion_seed=2026)
        lesion_fc = dataset_i['subject_maps']
        scores = dataset_i['scores']

        slnm = na.voxel_outcome_correlation(lesion_fc, scores[:, None]).flatten()
        gt_slnm.append(pearsonr(slnm, ground_truth_nii)[0])

        for j, pc_file in enumerate(pc1_3, start=1):
            pc = na.nifti_getdata(pc_file)
            gt_pc[j].append(pearsonr(ground_truth_nii, pc)[0])
            slnm_pc[j].append(pearsonr(slnm, pc)[0])

    recover_results[effect_size] = {
        'gt_slnm': gt_slnm,
        'gt_pc': gt_pc,
        'slnm_pc': slnm_pc
    }

# Plot Figures

In [ ]:
import pickle
effect_sizes = [0.0, 0.3, 0.6, 0.99]

with open('/Users/sasinm3/neuroimage_scripts/projects/stable_projects/symptom_lnm/results/pc1_3_results.pkl', 'rb') as f:
    recover_results = pickle.load(f)


In [ ]:
def half_violin_box(ax, data, position, color, width=0.3):
    vp = ax.violinplot(data, positions=[position], showextrema=False, widths=width)
    for body in vp['bodies']:
        m = np.mean(body.get_paths()[0].vertices[:, 0])
        body.get_paths()[0].vertices[:, 0] = np.clip(body.get_paths()[0].vertices[:, 0], -np.inf, m)
        body.set_facecolor(color)
        body.set_alpha(0.6)
        body.set_edgecolor('none')
    bp = ax.boxplot(data, positions=[position + width * 0.25], widths=width * 0.3,
                    patch_artist=True, showfliers=False,
                    medianprops=dict(color='black', linewidth=1.5),
                    whiskerprops=dict(color='black'),
                    capprops=dict(color='black'))
    for patch in bp['boxes']:
        patch.set_facecolor(color)
        patch.set_alpha(0.8)


fig, ax = plt.subplots(figsize=(8, 5))
plt.rcParams['font.family'] = 'Arial'

es_colors = ['#4472C4', '#F0A030', '#C44E52', '#8B1A1A']

for idx, effect_size in enumerate(effect_sizes):
    gt_slnm = recover_results[effect_size]['gt_slnm']
    half_violin_box(ax, gt_slnm, idx + 1, es_colors[idx], width=0.5)

ax.set_xticks(range(1, len(effect_sizes) + 1))
ax.set_xticklabels([f'$\eta^2$={es}' for es in effect_sizes], fontsize=18)
ax.set_ylabel('Correlation', fontsize=18)
ax.tick_params(axis = 'y', labelsize = 14)
ax.set_title('Ground Truth - sLNM Similarity', fontsize=14)
ax.spines[['top', 'right']].set_visible(False)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)

x = np.arange(1,4)
es_colors = ['#4472C4', '#F0A030', '#C44E52', '#8B1A1A']
pc_labels = [f'PC{j}' for j in range(1,4)]


# GT-PC
for j in range(1,4):
    gt = np.arctanh(np.abs(recover_results[effect_sizes[0]]['gt_pc'][j]))
    half_violin_box(axes[0], gt, j, '#5B8C5A', width=0.5)

axes[0].set_xticks(x)
axes[0].set_xticklabels(pc_labels, fontsize=18)
axes[0].set_title('Ground Truth', fontsize=18)
axes[0].set_ylabel('PC-Alignment', fontsize=18)
axes[0].set_ylim(-0.02, 1.5)
axes[0].tick_params(axis = 'y', labelsize = 14)


# sLNM-PC and Difference
spacing = 0.18
for idx, effect_size in enumerate(effect_sizes):
    for j in range(1,4):
        gt = np.arctanh(np.abs(recover_results[effect_size]['gt_pc'][j]))
        slnm = np.arctanh(np.abs(recover_results[effect_size]['slnm_pc'][j]))
        diff = np.array(slnm) - np.array(gt)

        pos = j + (idx - 1.5) * spacing
        half_violin_box(axes[1], slnm, pos, es_colors[idx], width=spacing * 0.9)
        half_violin_box(axes[2], diff, pos, es_colors[idx], width=spacing * 0.9)

    axes[1].plot([], [], 's', color=es_colors[idx], label=f'$\eta^2$={effect_size}', markersize=10)
    axes[2].plot([], [], 's', color=es_colors[idx], label=f'$\eta^2$={effect_size}', markersize=10)

axes[1].set_xticks(x)
axes[1].set_xticklabels(pc_labels, fontsize=18)
axes[1].set_title('sLNM', fontsize=18)
axes[1].set_ylabel('Pc-Alignment', fontsize=18)
axes[1].set_ylim(-0.02, 1.5)
axes[1].tick_params(axis = 'y', labelsize = 14)
axes[1].legend(fontsize=10)

axes[2].set_xticks(x)
axes[2].set_xticklabels(pc_labels, fontsize=18)
axes[2].set_title('sLNM - Ground Truth', fontsize=18)
axes[2].axhline(0, color='gray', linestyle='--', linewidth=0.8)
axes[2].legend(fontsize=10)
axes[2].tick_params(axis = 'y', labelsize = 14)
axes[2].set_ylabel('Difference in PC-Alignment', fontsize=18)

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd

rows = []

# Panel 1: Ground Truth (η²=0.0 only, since GT doesn't depend on effect size)
for j in range(1, 4):
    gt = np.arctanh(np.abs(recover_results[effect_sizes[0]]['gt_pc'][j]))
    rows.append({'panel': 'GT', 'effect_size': '-', 'PC': j,
                 'mean': np.mean(gt), 'std': np.std(gt, ddof=1), 'n': len(gt)})

# Panels 2 & 3: sLNM and difference, across effect sizes
for effect_size in effect_sizes:
    for j in range(1, 4):
        gt = np.arctanh(np.abs(recover_results[effect_size]['gt_pc'][j]))
        slnm = np.arctanh(np.abs(recover_results[effect_size]['slnm_pc'][j]))
        diff = slnm - gt
        for label, data in [('sLNM', slnm), ('sLNM-GT', diff)]:
            rows.append({'panel': label, 'effect_size': effect_size, 'PC': j,
                         'mean': np.mean(data), 'std': np.std(data, ddof=1), 'n': len(data)})

df = pd.DataFrame(rows)
print(df.to_string(index=False, float_format=lambda x: f'{x:.2f}'))

print()
for _, r in df.iterrows():
    print(f"{r['panel']:<8} η²={r['effect_size']!s:<5} PC{r['PC']}: "
          f"z = {r['mean']:.2f} ± {r['std']:.2f}")

In [ ]:
import numpy as np

np.random.seed(1)

def jt_statistic(groups):
    """Compute J = sum of pairwise Mann-Whitney counts in predicted order."""
    J = 0.0
    for i in range(len(groups)):
        for j in range(i+1, len(groups)):
            d = np.subtract.outer(groups[j], groups[i])
            J += np.sum(d > 0) + 0.5 * np.sum(d == 0)
    return J

def jt_permutation(groups, n_perm=10000, seed=42):
    """JT test with permutation-based p-value"""
    rng = np.random.default_rng(seed)
    
    # Observed J
    J_obs = jt_statistic(groups)
    
    # Pool data; remember group sizes
    pooled = np.concatenate(groups)
    sizes = [len(g) for g in groups]
    split_idx = np.cumsum(sizes)[:-1]
    
    # Build null distribution by shuffling group labels
    J_null = np.empty(n_perm)
    for p in range(n_perm):
        shuffled = rng.permutation(pooled)
        perm_groups = np.split(shuffled, split_idx)
        J_null[p] = jt_statistic(perm_groups)
    
    # One-sided p: proportion of null J >= observed
    # (+1 in numerator/denominator avoids p=0; Phipson & Smyth 2010)
    p_perm = (np.sum(J_null >= J_obs) + 1) / (n_perm + 1)
    
    return J_obs, J_null, p_perm

# Run on your data
print("sLNM-PC alignment trend across effect sizes (permutation, n=10000):")
for j in range(1, 4):
    groups = [np.abs(np.arctanh(recover_results[es]['slnm_pc'][j])) for es in effect_sizes]
    J_obs, J_null, p = jt_permutation(groups, n_perm=10000, seed=42)
    print(f"  PC{j}: J_obs = {J_obs:.0f}, "
          f"null mean = {J_null.mean():.0f}, "
          f"null std = {J_null.std():.0f}, p = {p:.2f}")

print("\nDifference (sLNM - GT) trend across effect sizes (permutation, n=10000):")
for j in range(1, 4):
    groups = [np.abs(np.arctanh(recover_results[es]['slnm_pc'][j])) -
              np.abs(np.arctanh(recover_results[es]['gt_pc'][j])) for es in effect_sizes]
    J_obs, J_null, p = jt_permutation(groups, n_perm=10000, seed=42)
    print(f"  PC{j}: J_obs = {J_obs:.0f}, "
          f"null mean = {J_null.mean():.0f}, "
          f"null std = {J_null.std():.0f}, p = {p:.2f}")

# Scatter Plot

In [ ]:


plt.rcParams['font.family'] = 'Arial'

row_colors = {1: '#4477AA', 2: '#228833', 3: '#AA3377'}
fig, axes = plt.subplots(3, 4, figsize=(20, 14))

for col, effect in enumerate(effect_sizes):
    for row, pc in enumerate([1, 2, 3]):
        ax = axes[row, col]
        x = np.array(recover_results[effect]['gt_pc'][pc])
        y = np.array(recover_results[effect]['slnm_pc'][pc])

        ax.scatter(x, y, alpha=0.5, s=30, color=row_colors[pc])
        ax.plot([-1, 1], [-1, 1], 'k--', alpha=0.5, linewidth=1.5)

        ax.set_xlim(-1, 1)
        ax.set_ylim(-1, 1)
        ax.set_yticks([-1.0,-0.5,0.0,0.5,1.0])
        ax.tick_params(axis='both', labelsize=16)
        ax.grid(alpha=0.25)
        ax.spines[['top', 'right']].set_visible(False)

        if row == 0:
            ax.set_title(f'η² = {effect}', fontsize=20)
        if col == 0:
            ax.set_ylabel('sLNM–PC correlation (r)', fontsize=16)
            ax.text(-0.28, 0.5, f'PC{pc}', transform=ax.transAxes,
                    fontsize=24, fontweight='bold', rotation=90,
                    va='center', ha='center')
        if row == 2:
            ax.set_xlabel('Ground truth–PC correlation (r)', fontsize=16)

plt.tight_layout()
plt.show()